This notebook assumes that it is run from docs/tutorials (as a current directory of jupyter lab).

In [ ]:
import arviz as az
import numpy as np
import random
import pandas as pd
import scipy.stats as stats
import xarray as xr
from matplotlib import pyplot as plt

# move to a directory to above access the privugger code
import os, sys
sys.path.append(os.path.join("../../"))

In [ ]:
# privugger library
import privugger as pv

from IPython.display import display, clear_output

$S$ - the number of students    
$C$ - the number of courses  
$k_i$ - the number of courses the $i$th student is registered in  
$l_j$ - the number of seats in $j$th course  (needs to be somewhat consistent with the above, there should be enough seats for the students)  
$e_{i,j}$ - The $i$th student is enrolled in $j$th course, $i=1..S$, $j=1..C$  
$c_{i,j}$ - the number of comments written by the $i$th student for the $j$th course

A student can only write a comment on the courses they are enrolled in.  
The comments of the student for one course are consecutive, and there is a small chance of interleaving comment block for courses from different students.  
The students work on the course evaluation in an unknown order. Discovering that order, would almost fully re-identify the authors of the comments.

# Synthetic Data Generation

Let's generate a synthetic dataset. 

In [ ]:
S = 300  # number of students in the programme
C = 40   # number of courses offered this term

We first generate the number $k$ of courses each student takes,
assuming each student registers in at most 5 courses at a time, 
with mean 3.3, so $p = \mu/n$. The two constants are based on our 
prior knowledge about the software study programs at the IT University of Copenhagen.

In [ ]:
k = np.random.binomial(n = 5, p = 3.3 / 5.0, size = S)
k

Now  generate the number of seats on each course, as we want to 
have both large and small courses.  A typical course takes 20 students (the number of seats is larger, but not all are taken).
It turns out that a Poisson distribution has too low variance for modeling the course size distribution at IT University.  
The smallest courses I get from Poisson are about 14 students, and the largest are ca. 34.  The spread at ITU is much larger.  Thus I tried with negative binomial, and shift the sample right by 15, to ensure that each course has at least 15 seats. I think about 15 is a minimum seat assignment at ITU typically.  Unlike Poisson, negative binomial allows choosing a different mean and standard deviation.


In [ ]:
μ_size = 30
σ_size = 50
r_size = μ_size * μ_size / (σ_size * σ_size - μ_size)
p_size = μ_size /(σ_size * σ_size)
# sample C courses, then fix everything by 4, so that the smallest courses have 4 students`
l = np.random.negative_binomial(n = r_size, p = p_size, size = C)
l = l + 15
l

Let's plot the distribution of course sizes (defined as number of seats offered).

In [ ]:
l_sorted = np.sort(l)
_ ,ax = plt.subplots (1, figsize = (8, 2))
ax.vlines(x = l_sorted, ymin = 0, ymax = 2, colors = 'gray', lw = .4, label = f"{len(l)} courses")

xs = np.linspace(0, l_sorted[-1], 60)
kde = stats.gaussian_kde(l_sorted).evaluate(xs)
ax.plot(xs, kde, color = 'C1', lw = 1.3, label = f"a simple KDE")

ax.set_title("Distribution of Course Sizes (capacity)")
ax.set_xlim(0, l.max()+5)
ax.set_ylim(0, kde.max()*1.07)
ax.set_xlabel(f"Course Size (smallest course {l.min()}, largest course {l.max()})")
ax.legend();

Check that there are enough seats for all students in the courses


In [ ]:
k_sum = k.sum() # the total number of registrations to execute
l_sum = l.sum() # the total number of course seats available
assert l_sum >= k_sum, f"There is only {l_sum} seats int the courses but students ask for {k_sum}"

Populate courses with students. We assign the students with more unassigned courses first, and the larger courses first. More precisely, we give these a probabilistic upper hand.

In [ ]:
e = np.zeros((S, C), dtype=int)
while k_sum > 0:
    p_enrollments = k / k_sum # prob. distribution of picking up a student
    p_seats = l / l_sum       # prob. distribution of picking up a course
    i = np.random.choice(S, p = p_enrollments)
    k[i] -= 1
    k_sum -= 1
    j = np.random.choice(C, p = p_seats)
    l[j] -= 1
    l_sum -= 1
    e[i,j] = 1
e

(NB. `e[i][j] = 1` is the $i$th student being enrolled in the $j$th course).

Plot of the actual registrations (the actual course sizes after the registration process).

In [ ]:
r_sorted = np.sort(np.array([np.sum(e[:,course]) for course in range(C)]))
r_sorted = r_sorted[r_sorted >= 3] # cancel courses below three students
_ ,ax = plt.subplots (1, figsize = (8, 2))
ax.vlines(x = r_sorted, ymin = 0, ymax = 2, colors = 'gray', lw = .4, label = f"{len(r_sorted)} courses")

r_xs = np.linspace(0, r_sorted[-1], 60)
r_kde = stats.gaussian_kde(r_sorted).evaluate(r_xs)
ax.plot(r_xs, r_kde, color = 'C1', lw = 1.3, label = f"a simple KDE")

ax.set_title("Distribution of Course Sizes (by actual registrations)")
ax.set_xlim(0, r_sorted.max()+5)
ax.set_ylim(0, r_kde.max()*1.07)
ax.set_xlabel(f"Course Size (smallest course {r_sorted.min()}, largest course {r_sorted.max()})")
ax.legend();

In [ ]:
# sanity check
# the largest course (r_sorted[-1]) has less students than the number of seats assigned to the largest course (l_sorted[-1])
assert l_sorted[-1] >= r_sorted[-1]

Reduce the number of courses to those that are actually running (some might have received no registrations)

In [ ]:
C = len(r_sorted)
C

We first generate the student comments for the courses they are enrolled in.
Assume that a typical student writes two comments per course.  We are not yet ordering these comments in any way.

In [ ]:
c = np.random.poisson(lam = 2, size=(S, C))
total_comments = c.sum()
total_comments

We shall now schedule writing the above comments in time.  Assume that there are $H$ hours for performing the evaluation. For simplicity, a student performs evaluation within an hour. Then $H = 250$ amounts to about 10 days (by decreasing this number we introduce more noise, as more students will write evaluations at the same time, and their indices will be conflated.

In [ ]:
H = 250 

Randomly assign the writing hour for each student (`h[t]` is the time when student evaluates).

In [ ]:
h = np.random.choice(range(H), size=S)
h

We are going to store the evaluation log visible in the online system in the `output` variable as quadruples: time, student, course, and a comment id. The loop below produces the output, for all the time interval, shuffling the students randomly.  Note that comments for a single course are not mixed between students (they receive consecutive ids).  Comments for different courses may interleave between students (actually a more leaking implementation would not mix courses, and we can also try that).

In [ ]:
output = []
available_comment_id = 0

for t in range(H):
    # students allocated to the current hour (the first component is the actual array)
    active_students = np.where(h == t)[0]
    # narrow down to students that have not written all their comments yet
    active_students = [ student_id for student_id in active_students if c[student_id,:].sum() > 0 ]
    
    while active_students:
        student_id = random.choice(active_students)
        # write comments in the order of courses (we take them in the order of increasing course ids)
        course_id = np.argmax(c[student_id, :] > 0)
        evaluations = [ (t, student_id, course_id, available_comment_id + comment_id) 
                        for comment_id 
                        in range(c[student_id, course_id]) ]
        output.extend(evaluations)
        available_comment_id += c[student_id, course_id]
        c[student_id, course_id] = 0
        if c[student_id, :].sum() == 0:
            active_students.remove(student_id)
        

An evaluation is a tuple: time, student, course, comment_id

In [ ]:
output = pd.DataFrame(output, columns = ['t', 'student', 'course', 'comment'])
output['position'] = range(len(output))
output.set_index(["position"], inplace = True)
output = xr.Dataset.from_dataframe(output)

In [ ]:
output

The entire generated data sets consist of the following:
* The `output` variable that links students, course, and comment_id
* The `e` array tracking which student is registered for what courses

In [ ]:
output, e

# Leakage Modeling

* We have now generated the data. What we want to do is to release the comment column.
* The attacker has access to student enrolement (the $e_{ij}$ array). 
* We want to measure leakage of information to the non-existant (explicitly) variable comment-author $a_{ik}$ that matches the student $i$ to a comment $k$. We want to find maximum leakage of this kind (find some individuals), ideally given an existing data set (so different from the ESORICS scenario).
* The PyMC model should have the data generated by the above process ($e_{i,j}$) and the time series of comments and course ids $\{i,j\}$.
* Let's start with prototyping the disclosure program to make things more concrete:

In [ ]:
len(output.course)

In [ ]:
def release(dataset):
    student_course     = np.array(list(set(zip(dataset.student.values, dataset.course.values)))) # why this is not from e_ijs? (not sure if this matters; but it seems to hint which students are considered as it does not include the students that have not written comments)
    sorted_student_ids = np.argsort(student_course[:, 0])
    student_course     = student_course[sorted_student_ids]

    comment_course     = np.array(list(zip(dataset.comment.values, dataset.course.values)))

    return student_course, comment_course
    

In [ ]:
student_course, comment_course = release(output)
print(str(student_course))

student -> course

Q: Why there is so much sorting in the second column below? Shouldn't the sorting be disrupted at most every courses?

In [ ]:
comment_course

comment -> course

The attacker wants to learn is: comment -> student (or student -> comment)

* I do not think we can use the above program as a release program. Will privugger handle it?
* What changes do we need to make? I could avoid sorting, I could assume we receive three numpy arrays, or one array with three columns; Unique/set is going to be hard to avoid. 

* Start with a naive attacker, some uniform weak priors
* Make a Boolean array students $\times$ comments (the $e$ array serves as a likelihood perhaps). This could allow to build a model without the timining information. But then how can we reason about leakage in this model? As a baseline for another model?
* Causal secondary information: consecutive ids.
* We could also have a prior with 0.5 in the comments array (we could infer the probability that the student has written a comment, and compare it with the ground truth).
* Consider how we can anonymize it somehow (DP or so)
* 250114
    * What's the minimal number of student-comment variables we need? (by looking at the registrations and counting)
    * Can we analyze for one student at a time? We did this in the netflix example. AW believes that this should give an underestimation of leakage, becase we have a global napsack problem, so maximizing likelihood for all student-comment links decreases chance for some local optima (it is like a table 

### Modeling Notes
  
* One way to do this: model everything with priors, compose with $e_ij$, this reduces uncertainty. So to simplify things $e_ij$ goes into the prior.
* How can the result of the analysis be used with the existing data set? We can also check the prediction error? A new kind of leakage maesure?
* We could difference from mode solution, or compare the mode posterior probability and the actual dataset probability; possibly also zoomed at the individual students, to see who is most vulnerable. 

# Privugger

In [ ]:
student_no = max(student_course [:,0])+1
course_no = max(student_course[:,1])
comment_no = max(comment_course[:,0])

We want to model a relation: Distrib(P(student_id x comment_id)), but it should be easier Dist(student_id x comment_id), or even student_id x Dist(Boolean)^comment_id

Assume that the number of students, course, and comment_no (get them from the observation and make constants in prior)

In [ ]:
c2s = pv.Uniform("c2s", lower=0.0, upper=1.0, num_elements= (comment_no, student_no))

The above seems extensional. As now we really have a 3D space ... I think there was a way to make it simpler, where observations where coming from the comments arrays only